# OpenAI ToolSearch 구현 — tools 배열을 고정해서 캐시 미스 0 만들기

도구가 수십 개 이상이면 전부 `tools`에 넣기엔 토큰이 아깝습니다.
그래서 "필요할 때 검색해서 쓰는" 방식(ToolSearch)을 만드는데, 문제가 하나 있습니다.
**찾은 도구를 `tools` 배열에 추가하는 순간 프롬프트 캐시가 깨집니다.**

이 노트북은 그 문제를 피하는 설계를 구현합니다.
`tools`는 `tool_search`(찾기)와 `tool_invoke`(실행) 딱 2개로 고정하고,
검색 결과(도구 스키마)는 대화 내용 쪽으로 돌려줍니다. 대화는 뒤에 쌓이기만 하므로 캐시가 안 깨집니다.

이 노트북에서 하는 것:
1. ToolSearch 구현 (2단계 검색 + 실행 게이트웨이)
2. 캐시의 실제 동작 성질 확인 (동일 요청 반복, 멀티턴 누적)
3. 본 실험: 도구 20개를 검색·실행하는 세션에서 캐시 재사용 ~80% 확인

설계 문서: `cc_agent_bible/md_group/openai-toolsearch-kv-cache.md`

## 배경 — 왜 tools를 안 바꾸면 캐시가 안 깨지나

OpenAI 프롬프트 캐싱의 규칙:
- 요청 앞부분이 이전 요청과 **완전히 같아야** 그 부분을 캐시에서 재사용합니다.
- 1024 토큰 이상부터 자동 적용되고, 128 토큰 단위로 캐시됩니다.
- 얼마나 재사용했는지는 응답의 `usage.input_tokens_details.cached_tokens`에 나옵니다.

요청은 이 순서로 만들어집니다:

```
[tools 배열] [시스템 지시문] [대화 내용 (계속 뒤에 쌓임)]
 ↑ 맨 앞. 여기가 바뀌면 뒤 전체가 캐시 미스
```

정리하면:

```
캐시를 깨는 것:     tools 배열 변경 (맨 앞이 바뀌므로)
캐시를 안 깨는 것:  대화 뒤에 붙는 모든 것 — 검색 결과, 스키마 텍스트, 실행 결과
```

그래서 전략은 하나입니다. **tools를 한 번 정하면 절대 바꾸지 않는다.**
도구 스키마는 `tool_search`의 결과(대화 내용)로 전달하고,
실행은 고정된 `tool_invoke`가 대신 해줍니다.

In [1]:
import json
import re
import time

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-5-nano"


## 1. 도구 레지스트리 — 클라이언트에만 있는 전체 도구 목록

도구 24개를 준비합니다. 이 목록은 **API에 보내지 않습니다.** (토큰 0)
모델은 이 도구들의 존재를 모르는 상태에서 시작하고, `tool_search`로만 찾을 수 있습니다.

- `description`은 "무엇을 하는 도구"보다 **"언제 쓰는 도구"** 중심으로 씁니다. 검색 품질이 여기서 결정됩니다.
- 실행 함수(handler)는 전부 가짜입니다. 실제 슬랙이나 캘린더에 연결하지 않습니다.
  이 노트북의 목적은 캐시 동작 확인이라서 실행 결과는 문자열이면 충분합니다.

In [2]:
def make_tool(name, description, params, handler=None, hint=""):
    """레지스트리 항목 생성. params = {인자이름: (타입, 설명)}, 전부 필수 인자로 취급.
    hint는 클로드코드의 searchHint에 해당하는 큐레이션 검색 힌트 (선택)."""
    properties = {k: {"type": t, "description": d} for k, (t, d) in params.items()}

    def default_handler(args, _name=name):
        return f"[가짜 실행 결과] {_name} 실행 완료 — 입력: {json.dumps(args, ensure_ascii=False)}"

    return {
        "name": name,
        "description": description,
        "search_hint": hint,
        "parameters": {"type": "object", "properties": properties, "required": list(params)},
        "handler": handler or default_handler,
    }


_ALL_TOOLS = [
    make_tool("slack_send", "슬랙 채널에 새 메시지를 보낼 때 사용. 알림, 공지, 결과 보고 전송.",
              {"channel": ("string", "채널 이름. 예: #general"), "text": ("string", "보낼 메시지 내용")},
              handler=lambda args: f"[가짜 실행 결과] {args['channel']} 채널에 메시지 전송 완료: \"{args['text']}\""),
    make_tool("slack_read", "슬랙 채널의 최근 메시지를 읽을 때 사용.",
              {"channel": ("string", "채널 이름"), "limit": ("integer", "가져올 메시지 개수")}),
    make_tool("slack_search", "슬랙 전체에서 키워드로 과거 메시지를 찾을 때 사용.",
              {"keyword": ("string", "검색 키워드")}),
    make_tool("calendar_create_event", "캘린더에 새 일정을 등록할 때 사용. 회의, 약속 생성.",
              {"title": ("string", "일정 제목"), "date": ("string", "날짜 YYYY-MM-DD"), "time": ("string", "시작 시각 HH:MM")},
              handler=lambda args: f"[가짜 실행 결과] 일정 등록 완료: {args['date']} {args['time']} \"{args['title']}\" (event_id=evt_1042)"),
    make_tool("calendar_list_events", "특정 날짜에 어떤 일정이 있는지 확인할 때 사용.",
              {"date": ("string", "날짜 YYYY-MM-DD")}),
    make_tool("calendar_delete_event", "등록된 일정을 취소할 때 사용.",
              {"event_id": ("string", "일정 ID")}),
    make_tool("gmail_send", "이메일을 보낼 때 사용.",
              {"to": ("string", "받는 사람 주소"), "subject": ("string", "제목"), "body": ("string", "본문")},
              hint="메일 이메일 전송 발송"),
    make_tool("gmail_search", "받은 메일함에서 메일을 찾을 때 사용.",
              {"query": ("string", "검색어")}),
    make_tool("gmail_read", "메일 한 통의 본문을 읽을 때 사용.",
              {"message_id": ("string", "메일 ID")}),
    make_tool("jira_create_issue", "지라에 새 이슈(티켓)를 만들 때 사용.",
              {"project": ("string", "프로젝트 키"), "title": ("string", "이슈 제목"), "description": ("string", "이슈 내용")},
              hint="지라 이슈 티켓 생성"),
    make_tool("jira_search_issues", "지라에서 이슈를 검색할 때 사용.",
              {"keyword": ("string", "검색 키워드")}),
    make_tool("jira_add_comment", "지라 이슈에 댓글을 달 때 사용.",
              {"issue_id": ("string", "이슈 ID"), "comment": ("string", "댓글 내용")}),
    make_tool("github_create_pr", "깃허브에 풀리퀘스트를 만들 때 사용.",
              {"repo": ("string", "저장소 이름"), "title": ("string", "PR 제목"), "branch": ("string", "브랜치 이름")}),
    make_tool("github_list_issues", "깃허브 저장소의 이슈 목록을 볼 때 사용.",
              {"repo": ("string", "저장소 이름")}),
    make_tool("github_merge_pr", "깃허브 풀리퀘스트를 머지할 때 사용.",
              {"repo": ("string", "저장소 이름"), "pr_number": ("integer", "PR 번호")}),
    make_tool("notion_create_page", "노션에 새 문서 페이지를 만들 때 사용.",
              {"title": ("string", "페이지 제목"), "content": ("string", "페이지 내용")}),
    make_tool("notion_search", "노션에서 문서를 검색할 때 사용.",
              {"keyword": ("string", "검색 키워드")},
              hint="노션 문서 페이지 검색"),
    make_tool("weather_get", "특정 도시의 현재 날씨를 확인할 때 사용.",
              {"city": ("string", "도시 이름. 예: 서울")},
              handler=lambda args: f"[가짜 실행 결과] {args['city']} 현재 날씨: 맑음, 기온 31도, 습도 62%",
              hint="날씨 기온 조회"),
    make_tool("translate_text", "문장을 다른 언어로 번역할 때 사용.",
              {"text": ("string", "번역할 문장"), "target_lang": ("string", "목표 언어. 예: en, ja")},
              hint="번역 언어 변환"),
    make_tool("currency_convert", "환율 기준으로 금액을 다른 통화로 바꿀 때 사용.",
              {"amount": ("number", "금액"), "from_currency": ("string", "원래 통화. 예: KRW"), "to_currency": ("string", "바꿀 통화. 예: USD")}),
    make_tool("db_query", "사내 데이터베이스에 SQL 조회를 실행할 때 사용.",
              {"sql": ("string", "실행할 SQL")},
              hint="데이터베이스 DB SQL 조회"),
    make_tool("file_read", "파일 내용을 읽을 때 사용.",
              {"path": ("string", "파일 경로")}),
    make_tool("file_write", "파일에 내용을 저장할 때 사용.",
              {"path": ("string", "파일 경로"), "content": ("string", "저장할 내용")}),
    make_tool("reminder_create", "지정한 시각에 알림을 만들 때 사용.",
              {"text": ("string", "알림 내용"), "when": ("string", "알림 시각 YYYY-MM-DD HH:MM")}),
]

REGISTRY = {t["name"]: t for t in _ALL_TOOLS}
print(f"레지스트리 도구 수: {len(REGISTRY)}개")
print(", ".join(REGISTRY))

레지스트리 도구 수: 24개
slack_send, slack_read, slack_search, calendar_create_event, calendar_list_events, calendar_delete_event, gmail_send, gmail_search, gmail_read, jira_create_issue, jira_search_issues, jira_add_comment, github_create_pr, github_list_issues, github_merge_pr, notion_create_page, notion_search, weather_get, translate_text, currency_convert, db_query, file_read, file_write, reminder_create


## 2. API에 선언하는 도구는 딱 2개

`tool_search`와 `tool_invoke`만 `tools`에 넣습니다. **이 배열은 세션 내내 바꾸지 않습니다.**

한 가지 짚을 점: `tool_invoke`의 `arguments`는 어떤 도구가 올지 몰라서 자유 형식 객체입니다.
그래서 서버가 해주는 엄격한(strict) 인자 검증은 못 씁니다. 검증은 4번 섹션에서 클라이언트가 직접 합니다.
이게 이 설계가 지불하는 대가 중 하나입니다.

In [3]:
TOOL_SEARCH_DEF = {
    "type": "function",
    "name": "tool_search",
    "description": ("작업에 필요한 도구를 찾는다. "
                    "query가 'select:이름1,이름2' 형식이면 해당 도구의 스키마를 바로 돌려주고, "
                    "일반 키워드면 후보 도구 목록을 돌려준다."),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": ("정확한 이름을 알면 'select:도구이름' (쉼표로 여러 개 가능). "
                                "모르면 조사를 뺀 명사 키워드를 공백으로 구분해 입력. 예: '슬랙 메시지 전송'"),
            }
        },
        "required": ["query"],
    },
    "strict": False,
}

TOOL_INVOKE_DEF = {
    "type": "function",
    "name": "tool_invoke",
    "description": "tool_search로 스키마를 확인한 도구를 실제로 실행한다. 모든 도구 실행은 이 통로로만 한다.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "실행할 도구 이름"},
            "arguments": {"type": "object", "description": "그 도구의 스키마에 맞춘 인자 객체"},
        },
        "required": ["name", "arguments"],
    },
    "strict": False,
}

FROZEN_TOOLS = [TOOL_SEARCH_DEF, TOOL_INVOKE_DEF]  # 절대 변경하지 않는 배열

## 3. tool_search 구현 — 입력 형태에 따른 동작

| 입력 | 동작 |
|---|---|
| `select:slack_send` | 재검색 없이 레지스트리에서 바로 찾아 **풀 스키마** 리턴 (쉼표로 여러 개 가능, 없는 이름은 제외하고 알림) |
| `slack_send` (이름 그대로) | 검색 생략, 즉시 그 도구의 풀 스키마 리턴 (fast path) |
| `슬랙 메시지 전송` | 키워드 점수를 매겨 **상위 5개를 이름+설명 카드로만** 리턴. 모델이 골라서 `select:`로 다시 호출 |
| `+슬랙 메시지` | `+`가 붙은 키워드는 필수 — 그 키워드가 있는 도구만 후보에 남김 |

카드에 스키마를 안 싣는 이유: 후보 5개의 풀 스키마는 세션 끝까지 대화에 남아 토큰을 차지합니다.
필요한 것만 골라 받는 쪽이 쌉니다.

점수 규칙은 클로드코드 실제 구현(`src/tools/ToolSearchTool/ToolSearchTool.ts:186-302`)을 이식했습니다.
도구 이름을 단어로 분해한 뒤 (slack_send → slack, send) 키워드마다 점수를 더합니다.

| 매치 위치 | 점수 |
|---|---|
| 이름 단어와 정확히 일치 | +10 |
| 이름 단어에 부분 포함 | +5 |
| (여태 0점일 때) 이름 전체 문자열에 포함 | +3 |
| search_hint (도구별 큐레이션 검색 힌트) | +4 |
| 설명(description) | +2 |

클로드코드와 다르게 둔 부분 두 가지:
- 설명 매치에 클로드코드는 단어 경계 정규식(`\b`)을 쓰지만, 한글은 조사가 붙어서("메시지를") `\b`가 안 맞습니다.
  그래서 한글이 든 키워드만 substring 매칭으로 처리합니다.
- 후보 카드 단계와 "1위 점수가 2위의 2배 이상이면 바로 스키마" 숏컷은 클로드코드에 없습니다.
  클로드코드는 검색 결과를 tool_reference 블록으로 리턴하면 Anthropic API가 서버쪽에서 풀 스키마로
  확장해 주기 때문에 그런 단계가 필요 없습니다. 그 서버 확장이 없는 OpenAI라서 이 프로토콜을 씁니다.

In [4]:
TOP_N = 5
DOMINANT_RATIO = 2.0  # 1위 점수가 2위의 2배 이상이면 바로 스키마 리턴


def name_parts(name):
    # 클로드코드 parseToolName 이식: snake_case와 CamelCase를 단어로 분해
    spaced = re.sub(r"([a-z])([A-Z])", r"\1 \2", name).replace("_", " ")
    return [p for p in spaced.lower().split() if p]


def term_matches_text(term, text):
    # 클로드코드는 단어 경계 정규식(\b)을 쓰지만 한글 조사("메시지를")에는 안 맞아서
    # 한글이 든 키워드는 substring으로 매칭한다
    if re.search(r"[가-힣]", term):
        return term in text
    return re.search(r"\b" + re.escape(term) + r"\b", text) is not None


def score_tool(terms, tool):
    # 클로드코드 searchToolsWithKeywords 스코어링 이식 (가중치 10/5/3/4/2)
    parts = name_parts(tool["name"])
    full = " ".join(parts)
    desc = tool["description"].lower()
    hint = tool.get("search_hint", "").lower()
    score = 0
    for term in terms:
        if term in parts:
            score += 10          # 이름 단어 정확 일치
        elif any(term in p for p in parts):
            score += 5           # 이름 단어 부분 포함
        if score == 0 and term in full:
            score += 3           # 이름 전체 문자열 (보조)
        if hint and term_matches_text(term, hint):
            score += 4           # 큐레이션 검색 힌트
        if term_matches_text(term, desc):
            score += 2           # 설명
    return score


def tool_has_term(tool, term):
    parts = name_parts(tool["name"])
    return (term in parts or any(term in p for p in parts)
            or term_matches_text(term, tool["description"].lower())
            or (tool.get("search_hint", "") and term_matches_text(term, tool["search_hint"].lower())))


def full_schema_text(names):
    blocks = [
        json.dumps(
            {"name": REGISTRY[n]["name"],
             "description": REGISTRY[n]["description"],
             "parameters": REGISTRY[n]["parameters"]},
            ensure_ascii=False, indent=2)
        for n in names
    ]
    return ("도구 스키마:\n" + "\n".join(blocks)
            + "\n\n이제 tool_invoke(name=도구이름, arguments=스키마에 맞는 인자 객체)로 실행하세요.")


def log_search_event(sess, summary):
    # 검색이 무엇을 찾았는지 요청별 로그(search_result 열)에 기록
    if sess is not None:
        sess.setdefault("search_events", []).append(summary)


def handle_tool_search(query, sess=None):
    query = query.strip()

    # 모드 A — select: 정확한 이름 조회 (클로드코드처럼 부분 성공 허용)
    if query.lower().startswith("select:"):
        names = [n.strip() for n in query[len("select:"):].split(",") if n.strip()]
        found = [n for n in names if n in REGISTRY]
        missing = [n for n in names if n not in REGISTRY]
        if not found:
            log_search_event(sess, "결과없음")
            return f"ERROR: 없는 도구 이름 {missing}. 키워드로 다시 검색하세요."
        note = f"\n\n(없는 이름이라 제외됨: {', '.join(missing)})" if missing else ""
        log_search_event(sess, "스키마:" + ",".join(found))
        return full_schema_text(found) + note

    q = query.lower()

    # fast path — 쿼리 전체가 도구 이름과 정확히 일치하면 즉시 스키마 (클로드코드 이식)
    if q in REGISTRY:
        log_search_event(sess, "스키마:" + q)
        return full_schema_text([q])

    # 모드 B — 키워드 검색. "+키워드"는 필수 조건 (클로드코드 이식)
    raw_terms = [t for t in re.split(r"\s+", q) if t]
    required = [t[1:] for t in raw_terms if t.startswith("+") and len(t) > 1]
    optional = [t for t in raw_terms if not t.startswith("+")]
    terms = required + optional if required else raw_terms

    candidates = list(REGISTRY.values())
    if required:
        candidates = [t for t in candidates if all(tool_has_term(t, r) for r in required)]

    scored = sorted(((score_tool(terms, t), t) for t in candidates), key=lambda x: -x[0])
    scored = [(s, t) for s, t in scored if s > 0]
    if not scored:
        log_search_event(sess, "결과없음")
        return "검색 결과 없음. 조사를 뺀 다른 키워드로 다시 검색하세요. 예: '메일 전송', '일정 등록'"

    top = scored[:TOP_N]
    # 숏컷 — 압도적 1위면 고르기 생략
    if len(top) == 1 or top[0][0] >= DOMINANT_RATIO * top[1][0]:
        name = top[0][1]["name"]
        log_search_event(sess, "스키마:" + name)
        return "1위 점수가 압도적이라 바로 스키마를 리턴합니다.\n\n" + full_schema_text([name])

    log_search_event(sess, "후보:" + ",".join(t["name"] for _, t in top))
    cards = "\n".join(f"{i + 1}. {t['name']} — {t['description']} (점수 {s})"
                      for i, (s, t) in enumerate(top))
    return ("후보 도구 목록 (점수순):\n" + cards
            + "\n\n필요한 도구를 모두 골라 tool_search(query=\"select:이름1,이름2\")로 다시 호출하세요."
            + "\n맞는 것이 없으면 다른 키워드로 재검색하세요.")

## 4. tool_invoke 구현 — 검증하고 실행

`tool_invoke`는 이름으로 레지스트리에서 도구를 찾아 실행하는 통로입니다.
실행 전에 인자를 클라이언트가 직접 검증합니다 (필수 인자 누락, 타입 오류, 스키마에 없는 인자).

검증에 실패하면 예외를 던지지 않고 **`ERROR:`로 시작하는 문자열을 결과로 돌려줍니다.**
모델이 그 에러를 읽고 인자를 고쳐서 다시 호출하게 만드는 방식입니다.
스키마를 조회하지 않고 기억만으로 호출한 경우도 여기서 걸러집니다.

In [5]:
TYPE_CHECK = {"string": str, "integer": int, "number": (int, float),
              "boolean": bool, "array": list, "object": dict}


def validate_args(parameters, args):
    errors = []
    props = parameters.get("properties", {})
    for required in parameters.get("required", []):
        if required not in args:
            errors.append(f"필수 인자 '{required}' 누락")
    for key, value in args.items():
        if key not in props:
            errors.append(f"스키마에 없는 인자 '{key}'")
        elif not isinstance(value, TYPE_CHECK.get(props[key]["type"], object)):
            errors.append(f"'{key}'는 {props[key]['type']} 타입이어야 함")
    return errors


def handle_tool_invoke(name, arguments, sess=None):
    tool = REGISTRY.get(name)
    if tool is None:
        return f"ERROR: '{name}' 도구는 없습니다. tool_search로 먼저 조회하세요."
    if isinstance(arguments, str):  # 모델이 객체 대신 JSON 문자열로 보낸 경우
        try:
            arguments = json.loads(arguments)
        except json.JSONDecodeError:
            return "ERROR: arguments가 올바른 JSON 객체가 아닙니다. 스키마에 맞춰 다시 호출하세요."
    errors = validate_args(tool["parameters"], arguments)
    if errors:
        return "ERROR: 인자 검증 실패 — " + "; ".join(errors) + ". 스키마에 맞춰 다시 호출하세요."
    return tool["handler"](arguments)

## 5. API 호출 없이 동작 먼저 확인

검색과 실행이 의도대로 도는지 순수 함수 수준에서 확인합니다.

In [6]:
print("─── 키워드 검색 (후보가 여럿 → 카드 목록) ───")
print(handle_tool_search("슬랙 메시지 전송"))

print()
print("─── 키워드 검색 (압도적 1위 → 바로 스키마) ───")
print(handle_tool_search("날씨")[:300], "...")

print()
print("─── select: 직접 조회 ───")
print(handle_tool_search("select:slack_send")[:300], "...")

print()
print("─── fast path: 이름 그대로 입력 → 즉시 스키마 ───")
print(handle_tool_search("slack_send")[:200], "...")

print()
print("─── +필수 연산자: '메일'이 있는 도구만 후보 ───")
print(handle_tool_search("+메일 전송")[:300], "...")

print()
print("─── select: 부분 성공 — 없는 이름은 제외하고 알림 ───")
print(handle_tool_search("select:slack_send,slack_broadcast")[-120:])

─── 키워드 검색 (후보가 여럿 → 카드 목록) ───
후보 도구 목록 (점수순):
1. slack_send — 슬랙 채널에 새 메시지를 보낼 때 사용. 알림, 공지, 결과 보고 전송. (점수 6)
2. slack_read — 슬랙 채널의 최근 메시지를 읽을 때 사용. (점수 4)
3. slack_search — 슬랙 전체에서 키워드로 과거 메시지를 찾을 때 사용. (점수 4)
4. gmail_send — 이메일을 보낼 때 사용. (점수 4)

필요한 도구를 모두 골라 tool_search(query="select:이름1,이름2")로 다시 호출하세요.
맞는 것이 없으면 다른 키워드로 재검색하세요.

─── 키워드 검색 (압도적 1위 → 바로 스키마) ───
1위 점수가 압도적이라 바로 스키마를 리턴합니다.

도구 스키마:
{
  "name": "weather_get",
  "description": "특정 도시의 현재 날씨를 확인할 때 사용.",
  "parameters": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "도시 이름. 예: 서울"
      }
    },
    "required": [
      "city"
    ]
 ...

─── select: 직접 조회 ───
도구 스키마:
{
  "name": "slack_send",
  "description": "슬랙 채널에 새 메시지를 보낼 때 사용. 알림, 공지, 결과 보고 전송.",
  "parameters": {
    "type": "object",
    "properties": {
      "channel": {
        "type": "string",
        "description": "채널 이름. 예: #general"
      },
      "text": {
        "type": "string",
    

In [7]:
print("─── 검증 실패 (text 누락) → 에러 리턴 → 모델이 읽고 고치게 됨 ───")
print(handle_tool_invoke("slack_send", {"channel": "#deploy"}))

print()
print("─── 검증 통과 → 실행 ───")
print(handle_tool_invoke("slack_send", {"channel": "#deploy", "text": "배포 완료"}))

print()
print("─── 없는 도구 → 프로토콜로 되돌리는 에러 ───")
print(handle_tool_invoke("slack_broadcast", {}))

─── 검증 실패 (text 누락) → 에러 리턴 → 모델이 읽고 고치게 됨 ───
ERROR: 인자 검증 실패 — 필수 인자 'text' 누락. 스키마에 맞춰 다시 호출하세요.

─── 검증 통과 → 실행 ───
[가짜 실행 결과] #deploy 채널에 메시지 전송 완료: "배포 완료"

─── 없는 도구 → 프로토콜로 되돌리는 에러 ───
ERROR: 'slack_broadcast' 도구는 없습니다. tool_search로 먼저 조회하세요.


## 6. 에이전트 루프와 캐시 측정

이제 실제 API를 부릅니다. 매 요청마다 `usage`에서 두 값을 기록합니다.

- `input_tokens`: 이번 요청의 전체 입력 토큰
- `input_tokens_details.cached_tokens`: 그중 캐시에서 재사용한 토큰


In [8]:
def new_session(tools, instructions, cache_key):
    return {"tools": list(tools), "instructions": instructions, "cache_key": cache_key,
            "input_list": [], "log": [], "turn": 0, "search_events": []}


def record_usage(sess, response, elapsed):
    usage = response.usage
    cached = getattr(usage.input_tokens_details, "cached_tokens", 0) or 0
    row = {"req": len(sess["log"]) + 1, "turn": sess["turn"], "input": usage.input_tokens,
           "cached": cached, "output": usage.output_tokens,
           "search": "-", "sec": round(elapsed, 1)}
    sess["log"].append(row)
    mark = "✅ HIT" if cached > 0 else "❌ MISS"
    print(f"    [요청 {row['req']:>2}] input={row['input']:>6}  cached={cached:>6}  {mark}"
          f"  ({row['sec']}초)")


def dispatch(sess, name, args):
    if name == "tool_search":
        return handle_tool_search(args.get("query", ""), sess)
    if name == "tool_invoke":
        return handle_tool_invoke(args.get("name", ""), args.get("arguments", {}), sess)
    if name in REGISTRY:
        return f"ERROR: '{name}'은 직접 호출할 수 없습니다. tool_invoke를 사용하세요."
    return f"ERROR: 알 수 없는 도구 '{name}'"


def run_turn(sess, user_msg, max_requests=8):
    sess["turn"] += 1
    print(f"\n👤 사용자: {user_msg}")
    sess["input_list"].append({"role": "user", "content": user_msg})

    for _ in range(max_requests):
        start = time.perf_counter()
        response = client.responses.create(
            model=MODEL,
            instructions=sess["instructions"],
            input=sess["input_list"],
            tools=sess["tools"],
            prompt_cache_key=sess["cache_key"],
        )
        record_usage(sess, response, time.perf_counter() - start)
        sess["input_list"] += response.output  # reasoning, function_call 등을 그대로 누적

        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            print(f"🤖 답변: {response.output_text}")
            return

        invoked = []
        for call in calls:
            args = json.loads(call.arguments)
            result = dispatch(sess, call.name, args)
            if call.name == "tool_invoke":
                invoked.append("실행:" + str(args.get("name", "?")))
            preview = call.arguments if len(call.arguments) <= 90 else call.arguments[:90] + "…"
            print(f"    🔧 {call.name}({preview})")
            print(f"       → {result.splitlines()[0][:90]}")
            sess["input_list"].append(
                {"type": "function_call_output", "call_id": call.call_id, "output": result})

        # 이번 요청에서 검색·실행된 도구를 해당 행의 search_result로 기록
        events = sess.get("search_events", []) + invoked
        sess["search_events"] = []
        if events:
            sess["log"][-1]["search"] = " · ".join(events)

    print("⚠️ 최대 요청 횟수 도달 — 턴 종료")


def print_log(title, sess):
    print(f"═══ {title} ═══")
    print(f"{'요청':>3} {'턴':>3} {'input':>7} {'cached':>7} {'적중률':>5} {'초':>6}  search_result")
    for r in sess["log"]:
        rate = f"{r['cached'] / r['input'] * 100:.0f}%" if r["input"] else "-"
        sr = r.get("search", "-")
        if len(sr) > 52:
            sr = sr[:52] + "…"
        print(f"{r['req']:>4} {r['turn']:>3} {r['input']:>7} {r['cached']:>7} {rate:>6} {r['sec']:>6}  {sr}")
    total_input = sum(r["input"] for r in sess["log"])
    total_cached = sum(r["cached"] for r in sess["log"])
    misses = sum(1 for r in sess["log"] if r["cached"] == 0)
    print(f"합계: 요청 {len(sess['log'])}회 | 입력 {total_input:,} 토큰 | "
          f"캐시에서 재사용 {total_cached:,} 토큰 ({total_cached / total_input * 100:.0f}%) | 미스 {misses}회")

## 7. 시스템 지시문

모델에게 2단계 프로토콜(검색 → 선택 → 스키마 수령 → 실행)을 가르치는 지시문입니다.

두 가지 의도적인 선택이 있습니다.
- **도구 이름 목록(카탈로그)을 지시문에 넣지 않았습니다.** 넣으면 모델이 검색을 건너뛰고 바로 `select:`로 가는 경우가 많아서, 키워드 검색 과정을 보여주기 어렵습니다. 실전에서는 넣는 쪽이 왕복을 줄여줍니다.
- 지시문을 일부러 넉넉하게 썼습니다. 캐싱이 1024 토큰부터 동작하므로, 고정 프리픽스(tools + 지시문)가 처음부터 그 기준을 넘게 만들기 위해서입니다.

In [9]:
RULES_V2 = """너는 사내 업무 비서다. 사용자의 요청을 도구를 사용해서 처리한다.

[도구 사용 규칙 — 반드시 이 순서대로 진행한다]
1. 너에게 항상 열려 있는 도구는 tool_search와 tool_invoke 두 개뿐이다.
2. 작업에 필요한 도구는 먼저 tool_search로 찾는다.
   - 도구 이름을 정확히 알면 tool_search(query="select:도구이름")으로 스키마를 바로 조회한다.
     여러 개가 필요하면 쉼표로 잇는다. 예: "select:slack_send,slack_read"
   - 이름을 모르면 조사를 뺀 명사 키워드를 공백으로 이어서 검색한다. 예: "슬랙 메시지 전송"
3. 검색 결과로 후보 도구 목록이 오면, 그중에서 필요한 도구를 모두 골라
   tool_search(query="select:...")로 다시 호출해서 스키마를 받는다.
   맞는 것이 없으면 키워드를 바꿔서 다시 검색한다.
4. 스키마를 받은 다음에만 tool_invoke(name=도구이름, arguments=인자객체)로 실행한다.
   arguments는 스키마의 properties와 required를 정확히 지킨다.
5. tool_invoke가 ERROR로 시작하는 결과를 돌려주면, 에러 내용을 읽고 인자를 고쳐서 다시 호출한다.
6. tool_search와 tool_invoke 외의 방법으로 도구를 호출하지 않는다.

[예시 흐름]
사용자: "팀에게 메일 보내줘"
1) tool_search(query="메일 전송")
2) 후보 목록 수신: gmail_send, gmail_search ...
3) tool_search(query="select:gmail_send")
4) gmail_send 스키마 수신
5) tool_invoke(name="gmail_send", arguments={"to": "...", "subject": "...", "body": "..."})
6) 실행 결과를 확인하고 사용자에게 한국어로 보고
"""

COMMON_POLICY = """[작업 정책]
- 날짜는 YYYY-MM-DD 형식, 시각은 HH:MM 24시간 형식으로 도구에 넘긴다. 연도가 없으면 2026년으로 본다.
- 메시지나 메일 본문은 사용자가 준 문구를 그대로 쓰고, 내용을 마음대로 추가하지 않는다.
- 삭제, 머지, 결제처럼 되돌리기 어려운 작업은 실행하기 전에 사용자에게 한 번 확인한다.
- 개인정보는 도구 인자에 꼭 필요한 경우에만 넣는다.
- 검색 결과가 비어 있으면 키워드를 바꿔서 한 번 더 검색하고, 그래도 없으면 없다고 보고한다.
- 한 요청에 여러 작업이 있으면 순서대로 하나씩 처리한다.
- 도구 실행 결과에 오류가 있으면 그 내용을 사용자에게 그대로 알린다.
- 실행하지 않은 작업을 했다고 말하지 않는다.
- 필요한 인자가 요청에 없으면 합리적인 값을 채우되, 최종 답변에서 그 사실을 밝힌다.
- 모든 시각은 한국 표준시(Asia/Seoul) 기준으로 해석한다.
- 도구를 실행하기 전에 스키마의 required 목록에 있는 인자가 전부 채워졌는지 확인한다.
- 같은 도구를 같은 인자로 두 번 연속 호출하지 않는다.
- 사용자가 시키지 않은 도구 실행은 하지 않는다.
- 도구 이름과 인자 이름은 스키마에 적힌 그대로 쓴다. 임의로 줄이거나 바꾸지 않는다.
- 답변 첫머리에 인사말, 감탄사, 이모지를 넣지 않는다.
- 작업이 끝나면 어떤 도구를 실행했고 결과가 무엇이었는지 한 줄로 요약해서 답한다.
- 최종 답변은 한국어로 간결하게 쓴다.
"""

INSTRUCTIONS = RULES_V2 + "\n" + COMMON_POLICY
print(f"지시문 길이: {len(INSTRUCTIONS)}자")

지시문 길이: 1709자


## 실험 — 본 실험: 도구 20개를 발견·실행하는 세션

실험 2-2는 발견하는 도구가 하나뿐이었습니다. 이 설계의 요점은
**"발견이 아무리 많아도 캐시가 안 깨진다"**이므로, 마지막으로 레지스트리 24개 중
20개를 실제로 검색·실행하는 세션을 돌립니다.

- 발견한 도구를 tools 배열에 장착하는 방식이었다면: 도구 발견 20회 = 구조적 미스 20회, 그때마다 쌓인 대화 전체 재캐싱
- 이 설계라면: tools 배열이 끝까지 2개, 구조적 미스는 최초 생성 1회뿐이어야 합니다

턴마다 서로 다른 도구 2~3개가 필요한 작업을 묶어서 시킵니다.
(삭제·머지 같은 작업은 정책상 모델이 확인을 요구할 수 있어서, 요청에 "승인됨"을 명시합니다.)

In [13]:
sess_many = new_session(tools=FROZEN_TOOLS, instructions=INSTRUCTIONS,
                        cache_key="toolsearch-many-tools-demo")

MANY_TOOL_TURNS = [
    "슬랙 #general에 '서버 점검 공지'라고 보내고, #dev 채널 최근 메시지 5개를 읽고, 슬랙 전체에서 '배포' 키워드를 검색해줘.",
    "7월 30일 14:00에 '분기 리뷰' 일정을 등록하고, 7월 30일 일정 목록을 확인하고, 일정 evt_1042를 취소해줘.",
    "kim@example.com에게 제목 '주간 보고', 본문 '첨부 확인 바랍니다'로 메일을 보내고, 받은 메일함에서 '계약서'를 검색하고, 메일 msg_100의 본문을 읽어줘.",
    "지라 PROJ 프로젝트에 '로그인 버그' 제목, '재현 절차 첨부' 내용으로 이슈를 만들고, '결제' 키워드로 이슈를 검색하고, 이슈 PROJ-1에 '확인했습니다' 댓글을 달아줘.",
    "깃허브 backend 저장소에 'fix: 오타 수정' 제목, hotfix 브랜치로 PR을 만들고, backend 저장소의 이슈 목록을 보고, backend 저장소 PR 42번을 머지해줘.",
    "노션에 '주간 보고' 제목, '이번 주 완료 항목 정리' 내용으로 페이지를 만들고, 노션에서 'OKR'을 검색해줘.",
    "서울 날씨를 확인하고, '배포가 완료되었습니다'를 영어로 번역하고, 100만 원이 몇 USD인지 환전 계산해줘.",
]

for msg in MANY_TOOL_TURNS:
    run_turn(sess_many, msg + " 모두 이미 승인된 작업이니 추가 확인 없이 바로 실행해줘.", max_requests=12)

print()
print_log("실험 3 — 도구 20개 발견 세션", sess_many)

# 이 세션에서 tool_invoke로 실행된 서로 다른 도구 수 집계
used = set()
for item in sess_many["input_list"]:
    item_type = getattr(item, "type", None) or (item.get("type") if isinstance(item, dict) else None)
    if item_type == "function_call" and getattr(item, "name", None) == "tool_invoke":
        try:
            used.add(json.loads(item.arguments).get("name"))
        except (json.JSONDecodeError, AttributeError):
            pass
print(f"\n이 세션에서 실행한 서로 다른 도구: {len(used)}개")
print(", ".join(sorted(used)))


👤 사용자: 슬랙 #general에 '서버 점검 공지'라고 보내고, #dev 채널 최근 메시지 5개를 읽고, 슬랙 전체에서 '배포' 키워드를 검색해줘. 모두 이미 승인된 작업이니 추가 확인 없이 바로 실행해줘.
    [요청  1] input=  1098  cached=     0  ❌ MISS  (2.6초)
    🔧 tool_search({"query":"슬랙 메시지 전송"})
       → 후보 도구 목록 (점수순):
    [요청  2] input=  1574  cached=  1280  ✅ HIT  (1.5초)
    🔧 tool_search({"query":"select:slack_send,slack_read,slack_search"})
       → 도구 스키마:
    [요청  3] input=  2094  cached=  1664  ✅ HIT  (5.1초)
    🔧 tool_invoke({"name":"slack_send","arguments":{"channel":"#general","text":"서버 점검 공지"}})
       → [가짜 실행 결과] #general 채널에 메시지 전송 완료: "서버 점검 공지"
    [요청  4] input=  2553  cached=  2432  ✅ HIT  (2.0초)
    🔧 tool_invoke({"name":"slack_read","arguments":{"channel":"#dev","limit":5}})
       → [가짜 실행 결과] slack_read 실행 완료 — 입력: {"channel": "#dev", "limit": 5}
    [요청  5] input=  2737  cached=  2560  ✅ HIT  (5.2초)
    🔧 tool_invoke({"name":"slack_search","arguments":{"keyword":"배포"}})
       → [가짜 실행 결과] slack_search 실행 완료 — 입력: {"keyword": "배포"}
    [요청 

도구를 20개 발견·실행하는 동안에도 tools 배열은 2개에서 변하지 않고,
미스는 최초 생성 + 서버 노이즈 소수뿐입니다. 장착 방식이었다면 이 세션에서만
구조적 미스가 20회 추가되고, 뒤로 갈수록(대화가 길수록) 재캐싱 비용이 커졌을 것입니다.
발견된 스키마들은 전부 대화 꼬리에 쌓여 다음 요청부터 캐시의 일부가 됩니다.

## 정리

- 캐시를 깨는 것은 검색도 실행도 아니고 **tools 배열 변경뿐**입니다. 배열을 고정하고 스키마를 대화 쪽으로 흘리면
  우리 쪽에서 캐시를 깰 일이 사라집니다.
- 그래도 남는 미스는 실험 1에서 분리한 세 가지(최초 생성, 쓰기 반영 지연, 서버 분산)이며
  전부 클라이언트 설계 밖의 요인입니다. 고정 프리픽스(tools + 시스템 지시문)를 1024보다
  수백 토큰 넉넉히 크게 유지하는 것만 챙기면 됩니다.
- 공짜는 아닙니다. 이 설계가 지불하는 대가:
  - 서버측 strict 인자 검증을 못 씁니다. 대신 클라이언트 검증 + 에러 리턴으로 모델이 스스로 고치게 합니다.
  - 키워드가 모호하면 후보를 고르는 왕복이 1회 추가됩니다. (압도적 1위 숏컷으로 일부 완화)
- 실무에서 섞어 쓰는 기준:
  - 항상 쓰는 핵심 도구는 처음부터 `tools`에 넣습니다. 최초 캐시 생성에 포함되므로 추가 비용이 없습니다.
  - 결제·삭제처럼 인자 정확성이 중요한 도구만 배열에 장착해서 strict 검증을 받고, 그때의 미스 1회는 의식하고 지불합니다.
  - 나머지 많은 도구는 전부 이 노트북의 디스패처 방식으로 처리합니다.
- 운영할 때는 `cached_tokens`를 계속 지켜보세요. 이 설계가 제대로 돌고 있다면 첫 요청 이후의 미스는 버그 신호입니다.
- 도구가 몇 개 안 되면(전체 컨텍스트의 10% 미만) 이 설계 자체가 과합니다. 그냥 전부 선언하는 게 쌉니다.